# **实验一**

# **0. 配置 Python 库**

In [ ]:
%pip install numpy matplotlib

# **1. 导入环境**
---
基本上都是内置的 Python 包，希望前面能安装好 `numpy` 和 `matplot` 这俩包。

In [2]:
from collections import deque
import heapq
import time
from typing import Tuple, List, Dict, Optional, Set, Iterable

# **2. 通用工具和框架**

In [ ]:
class SearchResult:
    """
    构建一个通用的节点记录类。
    可以用来回溯。
    """
    def __init__(
        self,
        found: bool,
        path: Optional[List],
        steps: int,
        expanded: int,
        time_s: float,
    ):
        self.found = found
        self.path = path or []
        self.steps = steps
        self.expanded = expanded
        self.time_s = time_s


def ReconstructPath(came_from: Dict, goal_state):
    path = []
    cur = goal_state
    while cur is not None:
        path.append(cur)
        cur = came_from.get(cur, None)
    path.reverse()
    return path


def BFS(start, goal, neighbors_fn):
    """
    广度优先搜索。
    neighbors_fn(state) -> iterable of neighbor states，
    返回 `SearchResult`。
    """

    t0 = time.perf_counter()
    q = deque([start])
    came_from = {start: None}
    visited: Set = {start}
    expanded = 0

    while q:
        state = q.popleft()
        expanded += 1
        if state == goal:
            path = ReconstructPath(came_from, state)
            return SearchResult(
                True, path, len(path) - 1, expanded, time.perf_counter() - t0
            )
        for nb in neighbors_fn(state):
            if nb not in visited:
                visited.add(nb)
                came_from[nb] = state
                q.append(nb)
    return SearchResult(False, None, 0, expanded, time.perf_counter() - t0)


def DFS(start, goal, neighbors_fn, depth_limit: int):
    """
    深度优先搜索（递归或显式栈），返回 SearchResult。
    depth_limit: 最大深度（步数），若找不到返回 not found。
    统计 expanded 节点数与最大 fringe（栈）大小。
    """
    t0 = time.perf_counter()
    stack = [(start, 0, None)]  # (state, depth, parent)
    came_from = {start: None}
    visited_depth = {start: 0}  # 记录到达某状态的最小深度以便剪枝
    expanded = 0
    max_fringe = 1

    while stack:
        state, depth, parent = stack.pop()
        # parent 已在 came_from 中设置（或 None）
        expanded += 1
        if state == goal:
            path = ReconstructPath(came_from, state)
            return SearchResult(
                True,
                path,
                len(path) - 1,
                expanded,
                time.perf_counter() - t0,
                max_fringe,
            )
        if depth >= depth_limit:
            if len(stack) > max_fringe:
                max_fringe = len(stack)
            continue
        # 生成邻居并压栈（为了更像递归 DFS，逆序压入）
        neighbors = list(neighbors_fn(state))
        # 可选：为了可重复性，按固定顺序处理
        for nb in reversed(neighbors):
            nd = depth + 1
            # 剪枝：如果之前以更浅深度到达过该状态，则跳过
            if nb in visited_depth and visited_depth[nb] <= nd:
                continue
            visited_depth[nb] = nd
            came_from[nb] = state
            stack.append((nb, nd, state))
        if len(stack) > max_fringe:
            max_fringe = len(stack)

    return SearchResult(False, None, 0, expanded, time.perf_counter() - t0, max_fringe)


def Astar(start, goal, neighbors_fn, heuristic_fn):
    """
    A* 搜索。
    neighbors_fn(state) -> iterable of (neighbor, cost) 或 neighbor（默认 cost=1），
    heuristic_fn(state) -> estimated cost to goal (must be admissible)，
    返回 `SearchResult`。
    """
    t0 = time.perf_counter()
    open_heap = []
    g_score = {start: 0}
    f_score = {start: heuristic_fn(start)}
    # heap 元素 (f, g, state, counter) 以避免相同 f,g 时比较 state
    counter = 0
    heapq.heappush(open_heap, (f_score[start], g_score[start], counter, start))
    came_from = {start: None}
    closed: Set = set()
    expanded = 0

    while open_heap:
        _, g, _, state = heapq.heappop(open_heap)
        if state in closed:
            continue
        expanded += 1
        if state == goal:
            path = ReconstructPath(came_from, state)
            return SearchResult(
                True, path, len(path) - 1, expanded, time.perf_counter() - t0
            )
        closed.add(state)

        for nb_item in neighbors_fn(state):
            # neighbors_fn 可以返回 (neighbor, cost) 或 neighbor
            if isinstance(nb_item, tuple) and len(nb_item) == 2:
                nb, cost = nb_item
            else:
                nb, cost = nb_item, 1

            tentative_g = g + cost
            if nb in closed and tentative_g >= g_score.get(nb, float("inf")):
                continue
            if tentative_g < g_score.get(nb, float("inf")):
                came_from[nb] = state
                g_score[nb] = tentative_g
                f = tentative_g + heuristic_fn(nb)
                counter += 1
                heapq.heappush(open_heap, (f, tentative_g, counter, nb))
    return SearchResult(False, None, 0, expanded, time.perf_counter() - t0)

# **3. 八数码**

In [ ]:
# 八数码状态表示：
# 使用长度为 9 的 tuple，按行主序 (r0c0..r0c2, r1c0..)
# 目标状态 `(1,2,3,4,5,6,7,8,0)`
GOAL_8PUZZLE = (1, 2, 3, 4, 5, 6, 7, 8, 0)


def IndexToMat(index):
    return divmod(index, 3)  # (row, col)


def MatToIndex(row, column):
    return row * 3 + column


def Parse8P(board):
    '''便捷函数：把 3x3 矩阵或字符串转为 tuple'''
    # board 可以是 list of lists 或 flat list 或 string "1 2 3 4 0 6 7 5 8"
    if isinstance(board, str):
        parts = board.split()
        arr = [int(x) for x in parts]
    elif isinstance(board, list) and len(board) == 3 and isinstance(board[0], list):
        arr = [x for row in board for x in row]
    else:
        arr = list(board)
    return tuple(arr)


def Neighbors8P(state: Tuple[int]) -> Iterable[Tuple[int]]:
    zero_idx = state.index(0)
    r, c = IndexToMat(zero_idx)
    moves = []
    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            nidx = MatToIndex(nr, nc)
            lst = list(state)
            lst[zero_idx], lst[nidx] = lst[nidx], lst[zero_idx]
            moves.append(tuple(lst))
    return moves


def Manhattan8P(state: Tuple[int], goal=GOAL_8PUZZLE) -> int:
    # 计算每个 tile 到目标位置的曼哈顿距离之和（不计空格）
    pos_goal = {val: i for i, val in enumerate(goal)}
    total = 0
    for i, val in enumerate(state):
        if val == 0:
            continue
        gi = pos_goal[val]
        r1, c1 = IndexToMat(i)
        r2, c2 = IndexToMat(gi)
        total += abs(r1 - r2) + abs(c1 - c2)
    return total

# **4. 传教士和野人**

In [ ]:
# 状态表示： (M_left, C_left, boat_side) boat_side: 1 表示船在左岸，0 表示在右岸
# 初始 (3,3,1) 目标 (0,0,0)
def ValidStateMC(state):
    M, C, boat = state
    # 各值范围
    if not (0 <= M <= 3 and 0 <= C <= 3 and boat in (0, 1)):
        return False
    # 左岸合法性：若左岸有传教士，传教士数 >= 野人数 或 野人数为0
    leftM, leftC = M, C
    rightM, rightC = 3 - M, 3 - C
    if leftM > 0 and leftM < leftC:
        return False
    if rightM > 0 and rightM < rightC:
        return False
    return True


def NeighborsMC(state):
    M, C, boat = state
    res = []
    # 船在左岸，载人到右岸；在右岸则相反
    # 可载组合： (1,0),(2,0),(0,1),(0,2),(1,1)
    moves = [(1, 0), (2, 0), (0, 1), (0, 2), (1, 1)]
    for dm, dc in moves:
        if boat == 1:
            nm = M - dm
            nc = C - dc
            nboat = 0
        else:
            nm = M + dm
            nc = C + dc
            nboat = 1
        ns = (nm, nc, nboat)
        if ValidStateMC(ns):
            res.append(ns)
    return res

# 启发式
def HeuristicMC(state):
    # 简单启发式：左岸剩余人数总和 / 船容量（2）向上取整
    M, C, boat = state
    remaining = M + C
    # 估计最少需要的渡河次数（每次最多带2人）
    est = (remaining + 1) // 2
    return est

# **5. 运行对比**

In [ ]:
# 老师的八数码的例子
start8P = Parse8P("1 2 3 4 0 6 7 5 8")
goal8P = GOAL_8PUZZLE

print("=== 8-Puzzle: start =", start8P, "goal =", goal8P)

# BFS（注意：八数码 BFS 可能非常慢，示例初始状态通常可解）
resultBfs8P = BFS(start8P, goal8P, Neighbors8P)
print(
    "BFS: found=",
    resultBfs8P.found,
    "steps=",
    resultBfs8P.steps,
    "expanded=",
    resultBfs8P.expanded,
    "time(s)=",
    round(resultBfs8P.time_s, 4),
)

# DFS，带有最高深度限制，避免等到猴年马月。
resultDfs8P = DFS(start8P, goal8P, Neighbors8P, depth_limit = 10)
print(
    "DFS (limit = ",
    resultDfs8P.depth_limit,
    "): found=",
    resultDfs8P.found,
    "steps=",
    resultDfs8P.steps,
    "expanded=",
    resultDfs8P.expanded,
    "time(s)=",
    round(resultDfs8P.time_s, 6),
    "max_fringe=",
    resultDfs8P.max_fringe,
)

# A*（曼哈顿）
res_astar_8p = Astar(
    start8P, goal8P, Neighbors8P, lambda s: Manhattan8P(s, goal8P)
)
print(
    "A*: found=",
    res_astar_8p.found,
    "steps=",
    res_astar_8p.steps,
    "expanded=",
    res_astar_8p.expanded,
    "time(s)=",
    round(res_astar_8p.time_s, 4),
)

# 若找到，打印路径长度与前几步（避免输出过长）
if res_astar_8p.found:
    print("A* path (len={}):".format(res_astar_8p.steps))
    for st in res_astar_8p.path[:6]:
        print(st)
    if res_astar_8p.steps > 6:
        print("... (total steps {})".format(res_astar_8p.steps))

# 传教士与野人实验
startMC = (3, 3, 1)
goalMC = (0, 0, 0)
print("\n=== Missionaries & Cannibals: start =", startMC, "goal =", goalMC)

res_bfs_mc = BFS(startMC, goalMC, NeighborsMC)
print(
    "BFS: found=",
    res_bfs_mc.found,
    "steps=",
    res_bfs_mc.steps,
    "expanded=",
    res_bfs_mc.expanded,
    "time(s)=",
    round(res_bfs_mc.time_s, 4),
)

res_astar_mc = Astar(startMC, goalMC, NeighborsMC, lambda s: HeuristicMC(s))
print(
    "A*: found=",
    res_astar_mc.found,
    "steps=",
    res_astar_mc.steps,
    "expanded=",
    res_astar_mc.expanded,
    "time(s)=",
    round(res_astar_mc.time_s, 4),
)

if res_astar_mc.found:
    print("A* path:", res_astar_mc.path)

=== 8-Puzzle: start = (1, 2, 3, 4, 0, 6, 7, 5, 8) goal = (1, 2, 3, 4, 5, 6, 7, 8, 0)
BFS: found= True steps= 2 expanded= 9 time(s)= 0.0
A*: found= True steps= 2 expanded= 3 time(s)= 0.0001
A* path (len=2):
(1, 2, 3, 4, 0, 6, 7, 5, 8)
(1, 2, 3, 4, 5, 6, 7, 0, 8)
(1, 2, 3, 4, 5, 6, 7, 8, 0)

=== Missionaries & Cannibals: start = (3, 3, 1) goal = (0, 0, 0)
BFS: found= True steps= 11 expanded= 15 time(s)= 0.0
A*: found= True steps= 11 expanded= 15 time(s)= 0.0001
A* path: [(3, 3, 1), (3, 1, 0), (3, 2, 1), (3, 0, 0), (3, 1, 1), (1, 1, 0), (2, 2, 1), (0, 2, 0), (0, 3, 1), (0, 1, 0), (1, 1, 1), (0, 0, 0)]
